# SQL reference 1: query fundamentals

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/sql/sql-reference-01-query-fundamentals.ipynb)

Logical evaluation order, the `SELECT` list, `DISTINCT`, predicates and pattern matching, sorting with NULL placement, `LIMIT`/`OFFSET`, parameter markers and Spark 4.0 pipe syntax. Practise it in [SQL practice 1: query fundamentals](sql-practice-01-query-fundamentals.ipynb).

*Last reviewed: November 2026, against Apache Spark 4.0 (Spark SQL).*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q pyspark~=4.0.0

## Setup and sample data

A local Spark session, the shared tables as temporary views: `customers`, `orders`, `order_items`, `products`, `events` and `employees` (customer 4 has no region, customers 7 and 8 never ordered, `SKU-4014` has no product row, one event is an exact retry, two pairs of employees tie on salary, Dana out-earns her manager and Jo has no department), and three helpers: `show(query)` runs a query and prints its rows untruncated, `plan(query)` returns what `EXPLAIN` prints, and `expected_error()` prints the error class of an example that fails on purpose.

In [ ]:
import os
import sys
import tempfile
import time

from pyspark.sql import SparkSession

os.environ["PYSPARK_PYTHON"] = sys.executable  # Python workers use this notebook's interpreter
os.environ["TZ"] = "UTC"  # the sample timestamps are UTC
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="pyspark-")  # scratch space for the files the lessons write

spark = (
    SparkSession.builder.master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")  # the default of 200 is far too many for tiny data
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

In [ ]:
# Shared sample tables as temporary views. Each statement is ordinary Spark SQL.
TABLES = {
    "customers": """CREATE OR REPLACE TEMP VIEW customers AS SELECT * FROM VALUES
  (1, 'Acme Corp',      'North', DATE'2024-01-03'),
  (2, 'Bright Retail',  'North', DATE'2024-02-11'),
  (3, 'Cyan Traders',   'South', DATE'2024-03-15'),
  (4, 'Delta Grocers',  NULL,    DATE'2024-05-02'),   -- region never arrived from the CRM
  (5, 'Echo Supplies',  'West',  DATE'2024-06-20'),
  (6, 'Foxtrot Foods',  'West',  DATE'2024-08-01'),
  (7, 'Gamma Sports',   'East',  DATE'2024-09-14'),   -- 7 and 8 have never ordered
  (8, 'Hotel Homeware', 'East',  DATE'2024-11-05')
AS customers(customer_id, name, region, signup_date)""",
    "orders": """CREATE OR REPLACE TEMP VIEW orders AS SELECT order_id, customer_id, status, order_ts, CAST(amount AS DECIMAL(12,2)) AS amount FROM VALUES
  (1,  1, 'paid',     TIMESTAMP'2026-01-05 09:15:00', 120.00),
  (2,  1, 'shipped',  TIMESTAMP'2026-01-05 10:05:00', 84.50),
  (3,  1, 'returned', TIMESTAMP'2026-01-05 11:40:00', 30.00),
  (4,  2, 'paid',     TIMESTAMP'2026-01-06 15:20:00', 260.00),
  (5,  2, 'placed',   TIMESTAMP'2026-01-07 08:05:00', 55.25),
  (6,  3, 'shipped',  TIMESTAMP'2026-01-07 17:45:00', 149.00),
  (7,  3, 'paid',     TIMESTAMP'2026-01-09 12:10:00', 310.00),
  (8,  4, 'returned', TIMESTAMP'2026-01-10 19:30:00', 89.90),
  (9,  4, 'shipped',  TIMESTAMP'2026-01-12 08:55:00', 1180.00),
  (10, 5, 'paid',     TIMESTAMP'2026-01-12 13:00:00', 74.00),
  (11, 6, 'placed',   TIMESTAMP'2026-01-13 10:25:00', 210.00),
  (12, 6, 'shipped',  TIMESTAMP'2026-01-14 00:30:00', 96.40)   -- just after midnight UTC
AS orders(order_id, customer_id, status, order_ts, amount)""",
    "order_items": """CREATE OR REPLACE TEMP VIEW order_items AS SELECT order_id, sku, qty, CAST(unit_price AS DECIMAL(8,2)) AS unit_price FROM VALUES
  (1, 'SKU-4001', 2, 12.50), (1, 'SKU-4007', 1, 95.00), (2, 'SKU-4002', 1, 34.50), (2, 'SKU-4009', 2, 25.00),
  (3, 'SKU-4002', 1, 30.00), (4, 'SKU-4003', 1, 149.00), (4, 'SKU-4004', 2, 55.50), (5, 'SKU-4005', 1, 55.25),
  (6, 'SKU-4003', 1, 149.00), (7, 'SKU-4006', 1, 189.00), (7, 'SKU-4008', 1, 121.00), (8, 'SKU-4010', 1, 89.90),
  (9, 'SKU-4011', 20, 59.00), (10, 'SKU-4012', 2, 37.00), (11, 'SKU-4013', 1, 210.00),
  (12, 'SKU-4014', 1, 41.40), (12, 'SKU-4005', 1, 55.00)
AS order_items(order_id, sku, qty, unit_price)""",
    "products": """CREATE OR REPLACE TEMP VIEW products AS SELECT sku, category, CAST(list_price AS DECIMAL(8,2)) AS list_price FROM VALUES
  ('SKU-4001', 'Kitchen', 12.50), ('SKU-4002', 'Kitchen', 34.50), ('SKU-4003', 'Garden', 149.00),
  ('SKU-4004', 'Garden', 55.50), ('SKU-4005', 'Pantry', 55.25), ('SKU-4006', 'Outdoor', 189.00),
  ('SKU-4007', 'Kitchen', 95.00), ('SKU-4008', 'Outdoor', 121.00), ('SKU-4009', 'Pantry', 25.00),
  ('SKU-4010', 'Garden', 89.90), ('SKU-4011', 'Pantry', 59.00), ('SKU-4012', 'Kitchen', 37.00),
  ('SKU-4013', 'Outdoor', 210.00)   -- SKU-4014 has no product row
AS products(sku, category, list_price)""",
    "events": """CREATE OR REPLACE TEMP VIEW events AS SELECT * FROM VALUES
  (1, 'view', TIMESTAMP'2026-01-06 09:00:00'), (1, 'add_to_cart', TIMESTAMP'2026-01-06 09:02:00'),
  (1, 'purchase', TIMESTAMP'2026-01-06 09:10:00'), (2, 'view', TIMESTAMP'2026-01-07 10:00:00'),
  (2, 'purchase', TIMESTAMP'2026-01-07 10:40:00'), (3, 'view', TIMESTAMP'2026-01-06 11:15:00'),
  (3, 'view', TIMESTAMP'2026-01-08 11:20:00'), (4, 'add_to_cart', TIMESTAMP'2026-01-09 20:05:00'),
  (4, 'purchase', TIMESTAMP'2026-01-10 08:30:00'), (5, 'view', TIMESTAMP'2026-01-11 14:45:00'),
  (5, 'view', TIMESTAMP'2026-01-12 09:00:00'), (5, 'purchase', TIMESTAMP'2026-01-12 12:30:00'),
  (5, 'view', TIMESTAMP'2026-01-12 09:00:00')   -- an SDK retry committed this event twice
AS events(user_id, event, ts)""",
    "employees": """CREATE OR REPLACE TEMP VIEW employees AS SELECT * FROM VALUES
  (1,  'Ada',    'Engineering', CAST(NULL AS INT), 185000, DATE'2018-03-01'),
  (2,  'Brett',  'Engineering', 1,    150000, DATE'2019-07-15'),
  (3,  'Chen',   'Engineering', 1,    150000, DATE'2020-01-06'),
  (4,  'Dana',   'Engineering', 2,    155000, DATE'2022-09-12'),   -- out-earns her manager
  (5,  'Eli',    'Sales',       1,    130000, DATE'2019-02-01'),
  (6,  'Farah',  'Sales',       5,    95000,  DATE'2021-05-17'),
  (7,  'Gus',    'Sales',       5,    95000,  DATE'2023-01-09'),
  (8,  'Hana',   'Finance',     1,    140000, DATE'2017-11-20'),
  (9,  'Ivan',   'Finance',     8,    88000,  DATE'2024-04-01'),
  (10, 'Jo',     NULL,          8,    70000,  DATE'2025-06-30')   -- department not assigned yet
AS employees(emp_id, name, dept, manager_id, salary, hire_date)""",
}
for statement in TABLES.values():
    spark.sql(statement)

import contextlib
import io
import logging
import re


def show(query, n=50, **args):
    """Run a query and print up to n rows without truncating values. Keyword arguments bind :name parameters."""
    spark.sql(query, args=args or None).show(n, truncate=False)


def plan(query, mode="formatted"):
    """The plan EXPLAIN prints for a query, as text."""
    return spark.sql(f"EXPLAIN {mode.upper()} {query}").first()[0]


@contextlib.contextmanager
def expected_error():
    """Run a block that is meant to fail and print the error class instead of a stack trace."""
    spark.sparkContext.setLogLevel("OFF")
    for logger in ("DataFrameQueryContextLogger", "SQLQueryContextLogger"):
        logging.getLogger(logger).setLevel(logging.CRITICAL)
    try:
        yield
    except Exception as error:
        lines = [line for line in str(error).splitlines() if line.strip()]
        message = next((line for line in lines if re.search(r"\[[A-Z][A-Z0-9_.]+\]", line)), lines[0] if lines else "")
        print(type(error).__name__ + ":", message.strip().split(". ")[0][:120])
    finally:
        spark.sparkContext.setLogLevel("ERROR")

## Contents

| section | entries |
| --- | --- |
| [How a query is evaluated](#how-a-query-is-evaluated) | [Logical evaluation order](#logical-evaluation-order) |
| [Choosing columns](#choosing-columns) | [`SELECT` list, aliases and `*`](#select-list-aliases-and-), [`DISTINCT`](#distinct) |
| [Filtering rows](#filtering-rows) | [Comparison and logical operators](#comparison-and-logical-operators), [Pattern matching: `LIKE`, `ILIKE`, `RLIKE`](#pattern-matching-like-ilike-rlike) |
| [Sorting and limiting](#sorting-and-limiting) | [`ORDER BY`, `LIMIT` and `OFFSET`](#order-by-limit-and-offset) |
| [Writing queries](#writing-queries) | [Parameter markers](#parameter-markers), [SQL pipe syntax (Spark 4.0)](#sql-pipe-syntax-spark-40) |

## How a query is evaluated

### Logical evaluation order

A `SELECT` statement is written in one order and **evaluated** in another. The logical order decides which names
each clause can see:

| step | clause | what it does | can reference |
| --- | --- | --- | --- |
| 1 | `FROM`, `JOIN` | build the input rows | tables, views, subqueries |
| 2 | `WHERE` | keep rows that satisfy a predicate | input columns only (no aggregates, no `SELECT` aliases in standard SQL) |
| 3 | `GROUP BY` | form groups | input columns and expressions |
| 4 | `HAVING` | keep groups | aggregates and grouping columns |
| 5 | window functions | compute over partitions of the remaining rows | everything above |
| 6 | `SELECT` | compute output expressions and aliases | everything above |
| 7 | `DISTINCT` | remove duplicate output rows | output columns |
| 8 | `ORDER BY` | sort | output aliases, positions, input columns |
| 9 | `LIMIT` / `OFFSET` | keep a slice | |

**Notes:** the engine may physically reorder work (push filters into scans, reorder joins) as long as the result
matches this logical order. Spark is more permissive than the standard: it allows `SELECT` aliases in `GROUP BY`,
`ORDER BY` and later `SELECT` items (lateral column aliases), but not in `WHERE`. Portable code repeats the expression
or uses a subquery or CTE.

In [ ]:
show("""
SELECT order_id, amount * 1.2 AS gross, gross - amount AS tax   -- lateral column alias: Spark, Snowflake, Databricks
FROM orders
ORDER BY gross DESC
LIMIT 3
""")
with expected_error():
    show("SELECT order_id, amount * 1.2 AS gross FROM orders WHERE gross > 300")   # WHERE runs before SELECT
show("SELECT * FROM (SELECT order_id, amount * 1.2 AS gross FROM orders) WHERE gross > 300 ORDER BY order_id")

## Choosing columns

### `SELECT` list, aliases and `*`

```sql
SELECT [DISTINCT] expr [[AS] alias], ...
SELECT * | table.* | * EXCEPT (col, ...)
```

| form | meaning |
| --- | --- |
| `expr AS alias` | name an output column; quote with backticks when it has spaces or is a keyword (`` `order date` ``) |
| `t.*` | all columns of one table in a join |
| `* EXCEPT (a, b)` | all columns but these (Spark, Databricks, BigQuery; Snowflake writes `* EXCLUDE`) |
| literals and expressions | `'EUR' AS currency`, `qty * unit_price AS line_total`, function calls |

**Notes:** avoid `SELECT *` in production code: a new upstream column silently changes your output schema, and
columnar engines read every column. Arithmetic on `DECIMAL` keeps exact decimal results; on `INT` it stays integral
(see [types](sql-07-types-functions-and-json.ipynb)).

In [ ]:
show("SELECT order_id, sku, qty * unit_price AS line_total, 'EUR' AS currency FROM order_items WHERE order_id <= 2")
show("SELECT * EXCEPT (order_ts, amount) FROM orders WHERE customer_id = 1")
show("SELECT name AS `customer name`, signup_date FROM customers LIMIT 2")

### `DISTINCT`

`SELECT DISTINCT` removes duplicate **output rows** (all selected columns together). NULLs compare as equal for this
purpose, so one NULL row survives. `count(DISTINCT col)` counts distinct non-NULL values.

**Notes:** `DISTINCT` is a full shuffle on a cluster. Reaching for it to "fix" duplicated join output usually hides a
join on the wrong grain; find the fan-out instead ([joins](sql-reference-04-joins.ipynb)).

In [ ]:
show("SELECT DISTINCT region FROM customers ORDER BY region")
show("SELECT DISTINCT customer_id, status FROM orders WHERE customer_id <= 2 ORDER BY customer_id, status")
show("SELECT count(*) AS rows, count(DISTINCT region) AS regions FROM customers")

## Filtering rows

### Comparison and logical operators

| operator | meaning | notes |
| --- | --- | --- |
| `=`, `<>` / `!=`, `<`, `<=`, `>`, `>=` | comparison | any comparison with NULL is NULL (unknown), so the row is dropped |
| `<=>` / `IS [NOT] DISTINCT FROM` | null-safe equality | see [NULLs](sql-reference-02-nulls-and-conditional-logic.ipynb) |
| `NOT`, `AND`, `OR` | logic | precedence is `NOT` > `AND` > `OR`: parenthesise mixed `AND`/`OR` |
| `x BETWEEN lo AND hi` | `lo <= x AND x <= hi` | **inclusive at both ends** |
| `x IN (a, b, c)` / `NOT IN` | membership | `NOT IN` with a NULL in the list returns no rows |

**Notes:** `BETWEEN` on timestamps is a classic bug: `order_ts BETWEEN '2026-01-05' AND '2026-01-07'` stops at
midnight on the 7th. Use a half-open range, `>= start AND < end + 1 day`.

In [ ]:
show("""
SELECT order_id, status, amount FROM orders
WHERE status = 'paid' OR status = 'shipped' AND amount > 200   -- AND binds first: all paid orders + big shipped ones
ORDER BY order_id
""")
show("""
SELECT order_id, status, amount FROM orders
WHERE (status = 'paid' OR status = 'shipped') AND amount > 200
ORDER BY order_id
""")
show("SELECT order_id, order_ts FROM orders WHERE order_ts BETWEEN '2026-01-05' AND '2026-01-07' ORDER BY order_id")
show("SELECT order_id, order_ts FROM orders WHERE order_ts >= '2026-01-05' AND order_ts < '2026-01-08' ORDER BY order_id")

### Pattern matching: `LIKE`, `ILIKE`, `RLIKE`

| predicate | matches |
| --- | --- |
| `s LIKE 'pat'` | `%` = any run of characters, `_` = exactly one; case-sensitive in Spark |
| `s ILIKE 'pat'` | case-insensitive `LIKE` (Spark 3.3+, Postgres, Snowflake) |
| `s LIKE 'a\_b' ESCAPE '\'` | match a literal `_` or `%` |
| `s LIKE ANY ('a%', 'b%')` / `LIKE ALL (...)` | any / all of several patterns (Spark, Snowflake) |
| `s RLIKE 'regex'` / `REGEXP` | Java regular expression, matching anywhere unless anchored with `^` / `$` |

**Notes:** a leading `%` prevents index use in row stores; in Spark it prevents nothing extra but still scans every
value. Regex syntax differs between engines (Java in Spark, RE2 in BigQuery, POSIX in Postgres).

In [ ]:
show("SELECT name FROM customers WHERE name LIKE '%Foods' OR name ILIKE '%retail%' ORDER BY name")
show("SELECT sku FROM products WHERE sku RLIKE '^SKU-40(0[1-3]|1[0-9])$' ORDER BY sku")
show("SELECT name FROM customers WHERE name LIKE ANY ('A%', 'E%', 'H%') ORDER BY name")

## Sorting and limiting

### `ORDER BY`, `LIMIT` and `OFFSET`

```sql
ORDER BY expr [ASC | DESC] [NULLS FIRST | NULLS LAST], ...
LIMIT n [OFFSET m]
```

| detail | Spark behaviour |
| --- | --- |
| default direction | `ASC` |
| NULL placement | `NULLS FIRST` for `ASC`, `NULLS LAST` for `DESC` (Postgres and Snowflake are the opposite) |
| sort keys | output aliases, column positions (`ORDER BY 2`), or any input expression |
| `OFFSET` | skip rows after sorting (Spark 3.4+) |

**Notes:** without `ORDER BY`, row order is undefined, and on a cluster it changes between runs. Always add a unique
tie-breaker (`ORDER BY amount DESC, order_id`) before `LIMIT`, or "top 3" is non-deterministic. Deep `OFFSET`
pagination re-reads all skipped rows; keyset pagination (`WHERE (ts, id) > (:last_ts, :last_id)`) does not.

In [ ]:
show("SELECT customer_id, region FROM customers ORDER BY region, customer_id")              # NULL region first
show("SELECT customer_id, region FROM customers ORDER BY region NULLS LAST, customer_id")
show("SELECT order_id, amount FROM orders ORDER BY amount DESC, order_id LIMIT 3")
show("SELECT order_id, order_ts FROM orders ORDER BY order_ts, order_id LIMIT 4 OFFSET 4")   # page 2 of 4 rows

## Writing queries

### Parameter markers

`spark.sql(query, args=...)` binds values to `:name` (dict) or `?` (list) markers. Values are bound as typed
literals, never pasted into the SQL text, so user input cannot change the statement (no SQL injection) and quoting is
handled for you. Markers stand for **values** only; table and column names go through `IDENTIFIER(:name)`.

In [ ]:
show("SELECT order_id, amount FROM orders WHERE status = :status AND amount >= :min ORDER BY order_id", status="paid", min=100)
spark.sql("SELECT count(*) AS n FROM IDENTIFIER(:t)", args={"t": "customers"}).show()

### SQL pipe syntax (Spark 4.0)

Spark 4.0 (and BigQuery) accept **pipe syntax**: a query starts with `FROM` and each `|>` step transforms the result
of the previous one, in the order the operations happen. It mixes freely with ordinary SQL.

| step | equivalent |
| --- | --- |
| `\|> WHERE pred` | `WHERE` (or `HAVING` after an aggregate) |
| `\|> SELECT ...` / `\|> EXTEND expr AS name` | projection / add columns |
| `\|> AGGREGATE agg GROUP BY keys` | `GROUP BY` |
| `\|> ORDER BY ...`, `\|> LIMIT n` | sorting and limiting |

In [ ]:
show("""
FROM orders
|> WHERE status <> 'returned'
|> EXTEND amount * 1.2 AS gross
|> AGGREGATE sum(gross) AS gross_total, count(*) AS orders GROUP BY customer_id
|> WHERE orders > 1
|> ORDER BY gross_total DESC
""")

**Next:** practise these entries in [SQL practice 1: query fundamentals](sql-practice-01-query-fundamentals.ipynb).